# LeWorldModel (LeWM) 复现 Notebook

完整复现 LeWM 训练流程，**兼容 Mac（MPS/CPU）和 Google Colab（T4 GPU）**。

| 环境 | `TASK` | `MAX_EPOCHS` | `BATCH_SIZE` | 精度 | 说明 |
|---|---|---|---|---|---|
| Mac 本地验证 | tworoom | 1 | 32 | fp16 | 跑通流程，几分钟 |
| Colab T4 复现 | tworoom | 3 | 64 | fp16 | 图小数据集小，T4 友好（**推荐**） |
| A100/H100 完整训练 | pusht | 100 | 128 | bf16 | 对齐源码，数小时 |

> 精度由 `device-config` 按 GPU 架构自动选择：Ampere+（sm≥8.0）用 bf16，T4（sm7.5）用 fp16。
> batch size 源码为 128，免费 T4 仅 15G 显存放不下，故降到 64（已在 `hparams` 注明）。

**使用方法**：Mac 本地在 LeWM 项目根目录激活 venv 后直接运行；Colab 见 Section 0。

In [1]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
IS_MAC   = sys.platform == 'darwin'

env_name = 'Google Colab' if IN_COLAB else ('Mac' if IS_MAC else 'Linux')
print(f'环境：{env_name}  |  Python {sys.version.split()[0]}')

环境：Google Colab  |  Python 3.12.13


## Section 0：Colab 环境安装

**仅在 Google Colab 中**取消注释并执行以下单元，Mac 本地跳过。

In [2]:
# ===== 仅在 Google Colab 中取消注释 =====

# 1. 克隆仓库
!git clone https://github.com/lucas-maes/le-wm.git
%cd le-wm

# 2. 安装依赖（训练不需要 env extra）
!pip install -q "stable-worldmodel[train]"

# 3. 验证
for pkg in ['stable_pretraining', 'stable_worldmodel', 'lightning']:
    try:
        __import__(pkg)
        print(f'OK  {pkg}')
    except ImportError:
        print(f'FAIL  {pkg}')

Cloning into 'le-wm'...
remote: Enumerating objects: 106, done.
remote: Total 106 (delta 0), reused 0 (delta 0), pack-reused 106 (from 1)
Receiving objects: 100% (106/106), 5.28 MiB | 3.67 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/le-wm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.2/305.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import wandb
# 如果是第一次使用，运行此行会提示输入 API Key，link: https://wandb.ai/authorize
if IN_COLAB:
    wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 591666522 (591666522-the-university-of-sydney) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Section 1：导入与设备配置

In [3]:
import json
from pathlib import Path

import torch
import torch.nn.functional as F
import lightning as pl
import stable_pretraining as spt
import stable_worldmodel as swm
import matplotlib.pyplot as plt
%matplotlib inline

# 从项目本地文件导入（jepa.py / module.py / utils.py）
from jepa import JEPA
from module import ARPredictor, Embedder, MLP, SIGReg
from utils import get_column_normalizer, get_img_preprocessor

print(f'torch      {torch.__version__}')
print(f'lightning  {pl.__version__}')

09:59:43 | INFO  | __init__.py | TensorFlow version 2.20.0 available.
09:59:43 | INFO  | __init__.py | JAX version 0.7.2 available.
09:59:49 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)
torch      2.11.0+cu128
lightning  2.6.5


In [ ]:
# ─────────────────────────────────────────────────────────
# 设备自动检测：CUDA > MPS (Apple Silicon) > CPU
# 精度按 GPU 架构自适应（仅硬件适配，不改训练语义）：
#   源码在 Ampere+（A100 等，sm≥8.0）用 bf16；
#   T4 是 Turing（sm7.5），Tensor Core 不支持 bf16，必须用 fp16 才走 Tensor Core，
#   否则 bf16 会退回普通 CUDA core，比 Mac 还慢。
# ─────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE      = 'cuda'
    ACCELERATOR = 'gpu'
    major, minor = torch.cuda.get_device_capability()
    PRECISION   = 'bf16-mixed' if major >= 8 else '16-mixed'  # T4(sm7.5) → fp16
    torch.backends.cudnn.benchmark = True          # 输入尺寸固定，自动挑最快卷积
    torch.set_float32_matmul_precision('high')
    print(f'GPU：{torch.cuda.get_device_name(0)}  (sm_{major}{minor})')
elif torch.backends.mps.is_available():
    DEVICE      = 'mps'
    ACCELERATOR = 'mps'
    PRECISION   = '16-mixed'
else:
    DEVICE      = 'cpu'
    ACCELERATOR = 'cpu'
    PRECISION   = '16-mixed'

print(f'设备：{DEVICE}  |  加速器：{ACCELERATOR}  |  精度：{PRECISION}')

## Section 2：训练超参

替代原项目的 Hydra yaml 配置（数值与 `config/train/lewm.yaml` 对齐）。
- **任务**：改 `TASK` 一行即可切换（默认 `tworoom`，免费 Colab 友好）
- **Mac 验证**：`MAX_EPOCHS=1`，`BATCH_SIZE=32`
- **Colab T4 复现**：`MAX_EPOCHS=3`，`BATCH_SIZE=64`（源码为 128，T4 显存放不下）

In [ ]:
# ══════════════════════════════════════════════════════════
# 任务选择：改这一行即可切换任务
# 可选：'pusht' | 'tworoom' | 'reacher' | 'cube'
TASK = 'tworoom'   # 免费 Colab 推荐 tworoom：图小、数据集小，T4 上能跑顺
# ══════════════════════════════════════════════════════════

# 各任务数据集配置（与 config/train/data/*.yaml 保持一致）
TASK_CONFIGS = {
    'pusht': {
        'description':   '机器人手臂推 T 形块到目标位置',
        'dataset_size':  '~43 GB（解压后）',
        'dataset_file':  'pusht_expert_train.h5',
        'hf_repo':       'quentinll/lewm-pusht',
        'hf_filename':   'pusht_expert_train.h5.zst',
        'keys_to_load':  ['pixels', 'action', 'proprio', 'state'],
        'keys_to_cache': ['action', 'proprio', 'state'],
        'norm_cols':     ['action', 'proprio', 'state'],
    },
    'tworoom': {
        'description':   '球穿过门到达指定位置',
        'dataset_size':  '~未知，需下载后确认',
        'dataset_file':  'tworoom.h5',
        'hf_repo':       'quentinll/lewm-tworooms',
        'hf_filename':   'tworoom.h5.zst',
        'keys_to_load':  ['pixels', 'action', 'proprio'],
        'keys_to_cache': ['action', 'proprio'],
        'norm_cols':     ['action', 'proprio'],
    },
    'reacher': {
        'description':   'DMC Reacher：机械臂末端到达目标点',
        'dataset_size':  '~未知，需下载后确认',
        'dataset_file':  'reacher.h5',
        'hf_repo':       'quentinll/lewm-reacher',
        'hf_filename':   'reacher.h5.zst',
        'keys_to_load':  ['pixels', 'action', 'observation'],
        'keys_to_cache': ['action', 'observation'],
        'norm_cols':     ['action', 'observation'],
    },
    'cube': {
        'description':   'OGBench：推方块到目标位置',
        'dataset_size':  '~未知，需下载后确认',
        'dataset_file':  'cube_single_expert.h5',
        'hf_repo':       'quentinll/lewm-cube',
        'hf_filename':   'cube_single_expert.h5.zst',
        'keys_to_load':  ['pixels', 'action', 'observation'],
        'keys_to_cache': ['action', 'observation'],
        'norm_cols':     ['action', 'observation'],
    },
}

cfg = TASK_CONFIGS[TASK]

# ── 根据目标环境修改这两行 ───────────────────────────────
MAX_EPOCHS = 3    # Mac 快速验证：1；Colab tworoom 复现：3
# 源码 batch=128，但 T4 仅 15G 显存（图统一 resize 到 224），128 易 OOM；
# Colab T4 用 64（实测约占 10/15G）。Mac 验证用 32。A100 等可恢复 128。
BATCH_SIZE = 64

# ── 固定超参（与原论文保持一致）────────────────────────────
IMG_SIZE      = 224
EMBED_DIM     = 192
HISTORY_SIZE  = 3
NUM_PREDS     = 1
NUM_STEPS     = HISTORY_SIZE + NUM_PREDS
FRAMESKIP     = 5

SIGREG_WEIGHT = 0.09
LR            = 5e-5
WEIGHT_DECAY  = 1e-3
GRAD_CLIP     = 1.0
TRAIN_SPLIT   = 0.9
SEED          = 3072

# DataLoader worker 数：Mac 用 0（HDF5/lance 多进程 fork 不安全）；
# 其他平台按实际 CPU 数，上限 4（Colab 免费机约 2 vCPU）。
NUM_WORKERS = 0 if IS_MAC else min(os.cpu_count() or 2, 4)

print(f'任务：{TASK}  —  {cfg["description"]}')
print(f'数据集：{cfg["dataset_file"]}  ({cfg["dataset_size"]})')
print(f'轮次：{MAX_EPOCHS}  批大小：{BATCH_SIZE}  worker：{NUM_WORKERS}  设备：{DEVICE}')

## Section 3：数据准备

数据集为 HDF5（`.h5`）格式，从 [HuggingFace 集合](https://huggingface.co/collections/quentinll/lewm) 下载（仓库内为 `.h5.zst` 压缩包）。

下面单元会打印对应任务的下载命令（国内用 `hf-mirror.com` 镜像），下载后用 `zstd -d` 解压到 `$STABLEWM_HOME`（Colab 默认 `/content/sample_data`，Mac 默认 `./data`）。

In [ ]:
import os

if IN_COLAB:
    STABLEWM_HOME = os.environ.get('STABLEWM_HOME', '/content/sample_data')
else:
    STABLEWM_HOME = os.environ.get('STABLEWM_HOME', os.path.abspath('./data'))

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
print(f'STABLEWM_HOME = {STABLEWM_HOME}')

archive_path = os.path.join(STABLEWM_HOME, cfg['hf_filename'])
dataset_path = os.path.join(STABLEWM_HOME, cfg['dataset_file'])

# 下载命令提示
if not os.path.exists(dataset_path) and not os.path.exists(archive_path):
    print(f'\n数据集不存在，请先下载：')
    print(f'wget -c "https://hf-mirror.com/datasets/{cfg["hf_repo"]}/resolve/main/{cfg["hf_filename"]}" \\')
    print(f'  -O {archive_path} \\')
    print(f'  --header="Authorization: Bearer YOUR_HF_TOKEN" \\')
    print(f'  --progress=bar:force')
    print(f'\n解压：')
    print(f'zstd -d {archive_path} -o {dataset_path} --progress')

# 自动解压（Colab 环境）
if os.path.exists(archive_path) and not os.path.exists(dataset_path):
    print('安装 zstd 解压工具...')
    os.system('apt-get install -y zstd')
    print(f'正在解压：{archive_path}')
    os.system(f'zstd -d "{archive_path}" -o "{dataset_path}"')
    print('解压完成！')

if os.path.exists(dataset_path):
    size_gb = os.path.getsize(dataset_path) / 1e9
    print(f'数据集已就绪：{dataset_path}  ({size_gb:.1f} GB)')
else:
    print(f'数据集尚未就绪，请先下载并解压。')

In [ ]:
import hdf5plugin
from stable_worldmodel.data.formats.hdf5 import HDF5Dataset

dataset = HDF5Dataset(
    path=dataset_path,
    num_steps=NUM_STEPS,
    frameskip=FRAMESKIP,
    keys_to_load=cfg['keys_to_load'],
    keys_to_cache=cfg['keys_to_cache'],
)
print(f'数据集大小：{len(dataset):,} 条')

ACTION_INPUT_DIM = FRAMESKIP * dataset.get_dim('action')
print(f'action_input_dim = {FRAMESKIP} × {dataset.get_dim("action")} = {ACTION_INPUT_DIM}')

transforms_list = [
    get_img_preprocessor(source='pixels', target='pixels', img_size=IMG_SIZE)
]
for col in cfg['norm_cols']:
    transforms_list.append(get_column_normalizer(dataset, col, col))

dataset.transform = spt.data.transforms.Compose(*transforms_list)
print(f'变换管道构建完成（归一化列：{cfg["norm_cols"]}）')

In [ ]:
# ── Train / Val 划分 + DataLoader ──────────────────────────
g = torch.Generator().manual_seed(SEED)
train_size = int(TRAIN_SPLIT * len(dataset))
val_size   = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(
    dataset, [train_size, val_size], generator=g
)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    persistent_workers=(NUM_WORKERS > 0),
    pin_memory=(DEVICE == 'cuda'),
)
if NUM_WORKERS > 0:
    loader_kwargs['prefetch_factor'] = 4   # 多预取，缓解 host I/O 瓶颈、喂饱 GPU

train_loader = torch.utils.data.DataLoader(
    train_set, shuffle=True, drop_last=True, generator=g, **loader_kwargs
)
val_loader = torch.utils.data.DataLoader(
    val_set, shuffle=False, drop_last=False, **loader_kwargs
)

print(f'训练集：{len(train_set):,}  验证集：{len(val_set):,}')
print(f'训练批次数：{len(train_loader)}')

## Section 4：模型构建

LeWM 四大组件：**Encoder** → **Projector** → **ARPredictor**（AdaLN-zero）← **Action Encoder** → **Pred Projector**

In [24]:
torch.manual_seed(SEED)

# 1. ViT-Tiny Encoder（不使用预训练权重，patch=14×14）
encoder = spt.backbone.utils.vit_hf(
    'tiny', patch_size=14, image_size=IMG_SIZE,
    pretrained=False, use_mask_token=False,
)

# 2. Projector：MLP + BatchNorm1d  →  encoder 输出映射到 latent
projector = MLP(
    input_dim=EMBED_DIM, hidden_dim=2048, output_dim=EMBED_DIM,
    norm_fn=torch.nn.BatchNorm1d,
)

# 3. ARPredictor：因果 Transformer，用 AdaLN-zero 以 action 为条件
predictor = ARPredictor(
    num_frames=HISTORY_SIZE,
    input_dim=EMBED_DIM, hidden_dim=EMBED_DIM, output_dim=EMBED_DIM,
    depth=6, heads=16, mlp_dim=2048, dim_head=64,
    dropout=0.1, emb_dropout=0.0,
)

# 4. Action Encoder：Conv1d + MLP  →  action 序列编码
action_encoder = Embedder(input_dim=ACTION_INPUT_DIM, emb_dim=EMBED_DIM)

# 5. Pred Projector：与 Projector 结构相同，作用于预测嵌入
pred_proj = MLP(
    input_dim=EMBED_DIM, hidden_dim=2048, output_dim=EMBED_DIM,
    norm_fn=torch.nn.BatchNorm1d,
)

# 组装 JEPA 世界模型
model = JEPA(
    encoder=encoder, predictor=predictor, action_encoder=action_encoder,
    projector=projector, pred_proj=pred_proj,
)

print('模型构建完成')

10:15:56 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
模型构建完成


In [25]:
total = sum(p.numel() for p in model.parameters())
print(f'总参数量：{total:,}  ({total/1e6:.1f}M)')
print()
for name, mod in [
    ('encoder',        encoder),
    ('projector',      projector),
    ('predictor',      predictor),
    ('action_encoder', action_encoder),
    ('pred_proj',      pred_proj),
]:
    n = sum(p.numel() for p in mod.parameters())
    print(f'  {name:<16}  {n:>9,}  ({n/1e6:.2f}M)')

总参数量：18,034,478  (18.0M)

  encoder           5,501,376  (5.50M)
  projector           792,768  (0.79M)
  predictor         10,791,360  (10.79M)
  action_encoder      156,206  (0.16M)
  pred_proj           792,768  (0.79M)


## Section 5：单批次前向验证

训练前用一个批次验证完整数据流，确认 tensor 形状和损失值正常。

In [26]:
model_dev = model.eval().to(DEVICE)

sample = next(iter(train_loader))
sample = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in sample.items()}

with torch.no_grad():
    # encode：pixels → CLS token → projector → emb
    out     = model_dev.encode(sample)
    emb     = out['emb']       # (B, T=4, D=192)
    act_emb = out['act_emb']   # (B, T=4, D=192)

    # predict：取前 history_size 步，预测后 1 步
    ctx_emb  = emb[:, :HISTORY_SIZE]      # (B, 3, 192)
    ctx_act  = act_emb[:, :HISTORY_SIZE]
    tgt_emb  = emb[:, NUM_PREDS:]         # (B, 3, 192) — 目标（偏移 1 步）
    pred_emb = model_dev.predict(ctx_emb, ctx_act)  # (B, 3, 192)

    # losses
    pred_loss   = (pred_emb - tgt_emb).pow(2).mean()
    sigreg_test = SIGReg().to(DEVICE)
    sig_loss    = sigreg_test(emb.transpose(0, 1))

print('前向传播验证通过')
print(f'  emb      {tuple(emb.shape)}')
print(f'  pred_emb {tuple(pred_emb.shape)}')
print(f'  tgt_emb  {tuple(tgt_emb.shape)}')
print(f'  pred_loss   = {pred_loss.item():.4f}')
print(f'  sigreg_loss = {sig_loss.item():.4f}')

/usr/local/lib/python3.12/dist-packages/lance/__init__.py:320: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn or forkserver instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lance/__init__.py:320: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn or forkserver instead.
  warnings.warn(
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'fo

前向传播验证通过
  emb      (128, 4, 192)
  pred_emb (128, 3, 192)
  tgt_emb  (128, 3, 192)
  pred_loss   = 0.0843
  sigreg_loss = 55.7440


## Section 6：训练

`LeWMModule` 实现 `train.py:lejepa_forward` 的完整逻辑，  
Loss = `pred_loss + 0.09 × sigreg_loss`。

In [27]:
class LeWMModule(pl.LightningModule):
    """复现 train.py:lejepa_forward 逻辑的 Lightning 训练模块。"""

    def __init__(self, model, history_size, num_preds, sigreg_weight,
                 lr, weight_decay, max_epochs):
        super().__init__()
        self.model         = model
        self.sigreg        = SIGReg(knots=17, num_proj=1024)
        self.history_size  = history_size
        self.num_preds     = num_preds
        self.sigreg_weight = sigreg_weight
        self.lr            = lr
        self.weight_decay  = weight_decay
        self.max_epochs    = max_epochs

    def _step(self, batch):
        batch['action'] = torch.nan_to_num(batch['action'], 0.0)
        out = self.model.encode(batch)
        emb     = out['emb']      # (B, T, D)
        act_emb = out['act_emb']

        ctx_emb  = emb[:, :self.history_size]
        ctx_act  = act_emb[:, :self.history_size]
        tgt_emb  = emb[:, self.num_preds:]
        pred_emb = self.model.predict(ctx_emb, ctx_act)

        pred_loss   = (pred_emb - tgt_emb).pow(2).mean()
        sigreg_loss = self.sigreg(emb.transpose(0, 1))
        loss        = pred_loss + self.sigreg_weight * sigreg_loss
        return loss, pred_loss, sigreg_loss

    def training_step(self, batch, batch_idx):
        loss, pred_loss, sigreg_loss = self._step(batch)
        self.log_dict({
            'train/loss': loss, 'train/pred_loss': pred_loss,
            'train/sigreg_loss': sigreg_loss,
        }, on_step=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, pred_loss, sigreg_loss = self._step(batch)
        self.log_dict({
            'val/loss': loss, 'val/pred_loss': pred_loss,
            'val/sigreg_loss': sigreg_loss,
        }, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        # LinearWarmup (10%) + CosineAnnealing，与原论文一致
        warmup = max(1, int(0.1 * self.max_epochs))
        sched = torch.optim.lr_scheduler.SequentialLR(
            opt,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(
                    opt, start_factor=0.1, end_factor=1.0, total_iters=warmup
                ),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    opt, T_max=max(1, self.max_epochs - warmup)
                ),
            ],
            milestones=[warmup],
        )
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'interval': 'epoch'}}

In [28]:
class LossHistory(pl.Callback):
    """记录每个 step 的训练指标和每个 epoch 的验证指标。"""

    def __init__(self):
        # step-level train metrics
        self.steps         = []
        self.train_loss    = []
        self.train_pred    = []
        self.train_sigreg  = []
        # epoch-level val metrics
        self.epochs        = []
        self.val_loss      = []
        self.val_pred      = []
        self.val_sigreg    = []
        # learning rate per epoch
        self.lr_history    = []

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        m = trainer.callback_metrics
        self.steps.append(trainer.global_step)
        self.train_loss.append(_get(m, 'train/loss'))
        self.train_pred.append(_get(m, 'train/pred_loss'))
        self.train_sigreg.append(_get(m, 'train/sigreg_loss'))

    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        if 'val/loss' not in m:
            return
        self.epochs.append(trainer.current_epoch)
        self.val_loss.append(_get(m, 'val/loss'))
        self.val_pred.append(_get(m, 'val/pred_loss'))
        self.val_sigreg.append(_get(m, 'val/sigreg_loss'))
        # 当前学习率
        opt = trainer.optimizers[0]
        self.lr_history.append(opt.param_groups[0]['lr'])

def _get(metrics, key):
    v = metrics.get(key)
    return float(v.detach().cpu()) if v is not None else float('nan')

In [ ]:
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import WandbLogger

model = model.cpu().train()

lewm = LeWMModule(
    model=model,
    history_size=HISTORY_SIZE,
    num_preds=NUM_PREDS,
    sigreg_weight=SIGREG_WEIGHT,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    max_epochs=MAX_EPOCHS,
)

loss_history = LossHistory()
progress_bar = TQDMProgressBar(refresh_rate=10)

wandb_logger = WandbLogger(
    entity=None,
    project='lewm',
    name=f'{TASK}-ep{MAX_EPOCHS}-bs{BATCH_SIZE}',
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=ACCELERATOR,
    devices=1,
    precision=PRECISION,
    gradient_clip_val=GRAD_CLIP,
    callbacks=[loss_history, progress_bar],
    logger=wandb_logger,
    enable_checkpointing=False,
    num_sanity_val_steps=1,
    enable_progress_bar=True,
)

print(f'开始训练：任务={TASK}  epoch={MAX_EPOCHS}  加速器={ACCELERATOR}  精度={PRECISION}')
trainer.fit(lewm, train_loader, val_loader)
print('训练完成')

In [ ]:
import numpy as np

def smooth(vals, w=20):
    """简单滑动平均平滑曲线。"""
    if len(vals) < w:
        return vals
    return np.convolve(vals, np.ones(w)/w, mode='valid')

steps  = loss_history.steps
epochs = loss_history.epochs if loss_history.epochs else list(range(1, len(loss_history.val_loss)+1))

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle('LeWM Training Dashboard', fontsize=14, fontweight='bold')

# ── 辅助函数 ─────────────────────────────────────────────
def plot_train(ax, vals, title, color='steelblue'):
    if not vals:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title); return
    ax.plot(steps, vals, alpha=0.25, color=color, linewidth=0.6, label='raw')
    s = smooth(vals)
    ax.plot(steps[len(steps)-len(s):], s, color=color, linewidth=1.5, label='smooth')
    ax.set_xlabel('Step'); ax.set_title(title); ax.grid(alpha=0.3); ax.legend(fontsize=8)

def plot_val(ax, vals, title, color='tomato', marker='o'):
    if not vals:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title); return
    ax.plot(epochs, vals, marker=marker, color=color, linewidth=2, markersize=5)
    ax.set_xlabel('Epoch'); ax.set_title(title); ax.grid(alpha=0.3)

# ── Row 0：总 Loss ───────────────────────────────────────
plot_train(axes[0,0], loss_history.train_loss,   'Train Loss (total)')
plot_val  (axes[0,1], loss_history.val_loss,     'Val Loss (total)')

# Train vs Val 总 Loss 叠加（epoch 级别）
ax = axes[0,2]
if loss_history.val_loss:
    # 把 train_loss 按 epoch 平均后叠加
    ep_boundaries = np.array_split(loss_history.train_loss, max(1, len(epochs)))
    train_ep = [np.nanmean(b) for b in ep_boundaries if len(b)]
    ep_x = list(range(1, len(train_ep)+1))
    ax.plot(ep_x,  train_ep,           'o-', color='steelblue', label='train', linewidth=1.5, markersize=4)
    ax.plot(epochs, loss_history.val_loss, 's-', color='tomato',    label='val',   linewidth=1.5, markersize=4)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
ax.set_title('Train vs Val Loss'); ax.grid(alpha=0.3)

# ── Row 1：Pred Loss ─────────────────────────────────────
plot_train(axes[1,0], loss_history.train_pred,  'Train Pred Loss', color='mediumseagreen')
plot_val  (axes[1,1], loss_history.val_pred,    'Val Pred Loss',   color='darkorange')

# Pred Loss 趋势
ax = axes[1,2]
if loss_history.val_pred:
    ep_boundaries = np.array_split(loss_history.train_pred, max(1, len(epochs)))
    train_ep = [np.nanmean(b) for b in ep_boundaries if len(b)]
    ep_x = list(range(1, len(train_ep)+1))
    ax.plot(ep_x,   train_ep,                'o-', color='mediumseagreen', label='train', linewidth=1.5, markersize=4)
    ax.plot(epochs, loss_history.val_pred,   's-', color='darkorange',     label='val',   linewidth=1.5, markersize=4)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
ax.set_title('Train vs Val Pred Loss'); ax.grid(alpha=0.3)

# ── Row 2：SIGReg Loss / LR / 分布 ──────────────────────
plot_train(axes[2,0], loss_history.train_sigreg, 'Train SIGReg Loss', color='mediumpurple')
plot_val  (axes[2,1], loss_history.val_sigreg,   'Val SIGReg Loss',   color='slateblue')

# 学习率曲线
ax = axes[2,2]
if loss_history.lr_history:
    ax.plot(epochs, loss_history.lr_history, color='goldenrod', linewidth=2, marker='.')
    ax.set_xlabel('Epoch'); ax.set_ylabel('LR'); ax.set_title('Learning Rate Schedule')
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Learning Rate Schedule')

plt.tight_layout()
plt.savefig('training_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print('已保存 training_dashboard.png (300 dpi)')

## Section 7：Checkpoint 保存与验证

保存两种格式（与原项目完全兼容）：
- `lewm_object.ckpt`：完整 Python 对象，`eval.py` 和 `swm.policy.AutoCostModel` 使用
- `weights.pt`：纯 state dict，HuggingFace 发布格式

In [ ]:
ckpt_dir = Path(STABLEWM_HOME) / 'checkpoints' / TASK
ckpt_dir.mkdir(parents=True, exist_ok=True)

object_path  = ckpt_dir / 'lewm_object.ckpt'
weights_path = ckpt_dir / 'weights.pt'

torch.save(lewm.model.cpu(), object_path)
print(f'完整对象已保存：{object_path}')

torch.save(lewm.model.state_dict(), weights_path)
print(f'权重已保存：    {weights_path}')

config_dict = {
    'task':           TASK,
    'encoder':        {'size': 'tiny', 'patch_size': 14, 'image_size': IMG_SIZE},
    'predictor':      {'num_frames': HISTORY_SIZE, 'input_dim': EMBED_DIM,
                       'hidden_dim': EMBED_DIM, 'output_dim': EMBED_DIM,
                       'depth': 6, 'heads': 16, 'mlp_dim': 2048, 'dim_head': 64},
    'action_encoder': {'input_dim': ACTION_INPUT_DIM, 'emb_dim': EMBED_DIM},
    'projector':      {'input_dim': EMBED_DIM, 'hidden_dim': 2048, 'output_dim': EMBED_DIM},
    'pred_proj':      {'input_dim': EMBED_DIM, 'hidden_dim': 2048, 'output_dim': EMBED_DIM},
}
with open(ckpt_dir / 'config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)
print(f'配置已保存：    {ckpt_dir / "config.json"}')

In [ ]:
# 加载验证：从 _object.ckpt 恢复模型，执行一次前向传播
loaded = torch.load(object_path, map_location='cpu', weights_only=False)
loaded.eval()

test_batch = {k: v[:2] if torch.is_tensor(v) else v
              for k, v in next(iter(val_loader)).items()}

with torch.no_grad():
    result = loaded.encode(test_batch)
    print(f'加载验证通过  emb 形状：{tuple(result["emb"].shape)}')

print()
print('要在 eval.py 中使用此 checkpoint，执行：')
print(f'  python eval.py --config-name=pusht.yaml policy=checkpoints/notebook_run/lewm')

## Section 8：在 Colab 上复现（tworoom，推荐流程）

1. 上传此 notebook 到 Google Colab，**Runtime → Change runtime type → T4 GPU**
2. 取消注释并运行 **Section 0** 安装单元（含 `wandb.login()`）
3. **Section 2** 保持默认即可（已为 Colab 调好）：
   ```python
   TASK       = 'tworoom'   # 图小、数据集小，T4 友好
   MAX_EPOCHS = 3
   BATCH_SIZE = 64          # 源码 128，T4 15G 显存放不下
   ```
   精度无需手动设置——`device-config` 检测到 T4(sm7.5) 会自动用 fp16。
4. **Section 3** 按打印出的命令下载并解压 `tworoom.h5.zst`
5. **Runtime → Run all** 顺序执行

**换更大算力时**：在 A100/H100 上可把 `TASK='pusht'`、`MAX_EPOCHS=100`、`BATCH_SIZE=128` 对齐源码完整训练，精度会自动切回 bf16。

**Checkpoint 位置**：`$STABLEWM_HOME/checkpoints/<TASK>/`（见 Section 7）。

## Resources

### 释放系统 RAM 和 GPU 显存

在训练或推理过程中，如果遇到内存不足的问题，可以使用以下代码清除系统 RAM 中不再使用的变量，并清除 PyTorch 缓存的 GPU 显存。

```python
import torch
import gc

# 释放 GPU 显存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cache has been cleared.")
else:
    print("No CUDA-enabled GPU found. No GPU memory to clear.")

# 释放系统 RAM
# 删除不再使用的变量，例如 del model, del data_loader
# ...

gc.collect()
print("System RAM garbage collection performed.")
```

In [32]:
import torch
import gc

# 释放 GPU 显存
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cache has been cleared.")
else:
    print("No CUDA-enabled GPU found. No GPU memory to clear.")

# 释放系统 RAM
# 删除不再使用的变量，例如 del model, del data_loader
# ...

gc.collect()
print("System RAM garbage collection performed.")

GPU memory cache has been cleared.
System RAM garbage collection performed.
